In [0]:
-- Business Question 2: How does weather affect taxi demand and trip behavior?
-- Grain: Temperature band × precipitation flag × weather condition × hour
-- Weather exposure is calculated from actual hourly Silver Weather observations.

WITH silver_weather_categorized AS (
    SELECT
        weather_datetime,

        CASE
            WHEN temperature_c < 0 THEN 'Below 0 C'
            WHEN temperature_c < 10 THEN '0-10 C'
            WHEN temperature_c < 20 THEN '10-20 C'
            WHEN temperature_c < 30 THEN '20-30 C'
            ELSE '30+ C'
        END AS temperature_band,

        CASE
            WHEN precipitation_mm > 0 THEN TRUE
            ELSE FALSE
        END AS precipitation_flag,

        CASE
            WHEN weather_code = 0 THEN 'Clear'
            WHEN weather_code IN (1, 2, 3) THEN 'Cloudy'
            WHEN weather_code IN (45, 48) THEN 'Fog'
            WHEN weather_code BETWEEN 51 AND 67 THEN 'Rain'
            WHEN weather_code BETWEEN 71 AND 77 THEN 'Snow'
            WHEN weather_code BETWEEN 80 AND 82 THEN 'Rain'
            WHEN weather_code BETWEEN 85 AND 86 THEN 'Snow'
            WHEN weather_code BETWEEN 95 AND 99 THEN 'Thunderstorm'
            ELSE 'Other'
        END AS weather_condition,

        CASE
            WHEN precipitation_mm > 0 THEN 'Rain'
            ELSE 'No Rain'
        END AS rain_condition,

        HOUR(weather_datetime) AS hour

    FROM nyc_mobility.silver.weather
),

weather_exposure AS (
    SELECT
        temperature_band,
        precipitation_flag,
        weather_condition,
        rain_condition,
        hour,
        COUNT(*) AS weather_hours

    FROM silver_weather_categorized

    GROUP BY
        temperature_band,
        precipitation_flag,
        weather_condition,
        rain_condition,
        hour
),

trip_weather AS (
    SELECT
        w.temperature_band,
        w.is_raining AS precipitation_flag,
        w.weather_condition,

        CASE
            WHEN w.is_raining = TRUE THEN 'Rain'
            ELSE 'No Rain'
        END AS rain_condition,

        h.hour_of_day AS hour,
        t.trip_count,
        t.fare_amount,
        t.trip_distance,
        t.trip_duration_minutes,
        t.total_amount

    FROM nyc_mobility.gold.fact_trip AS t

    JOIN nyc_mobility.gold.dim_weather AS w
        ON t.weather_key = w.weather_key

    JOIN nyc_mobility.gold.dim_hour AS h
        ON t.pickup_hour_key = h.hour_key
),

trip_metrics AS (
    SELECT
        temperature_band,
        precipitation_flag,
        weather_condition,
        rain_condition,
        hour,

        SUM(trip_count) AS trip_count,
        SUM(fare_amount) AS total_fare,
        AVG(trip_distance) AS avg_trip_distance,
        AVG(trip_duration_minutes) AS avg_trip_duration,
        AVG(trip_duration_minutes) AS avg_trip_duration_minutes,
        AVG(fare_amount) AS avg_fare,
        AVG(total_amount) AS avg_total_amount

    FROM trip_weather

    GROUP BY
        temperature_band,
        precipitation_flag,
        weather_condition,
        rain_condition,
        hour
)

SELECT
    t.temperature_band,
    t.precipitation_flag,
    t.weather_condition,
    t.rain_condition,
    t.hour,
    t.trip_count,
    t.total_fare,

    ROUND(t.avg_trip_distance, 2) AS avg_trip_distance,
    ROUND(t.avg_trip_duration, 1) AS avg_trip_duration,
    ROUND(t.avg_trip_duration_minutes, 2) AS avg_trip_duration_minutes,
    ROUND(t.avg_fare, 2) AS avg_fare,
    ROUND(t.avg_total_amount, 2) AS avg_total_amount,

    w.weather_hours,

    ROUND(
        t.trip_count * 1.0 / NULLIF(w.weather_hours, 0),
        2
    ) AS avg_trips_per_hour

FROM trip_metrics AS t

LEFT JOIN weather_exposure AS w
    ON t.temperature_band = w.temperature_band
    AND t.precipitation_flag = w.precipitation_flag
    AND t.weather_condition = w.weather_condition
    AND t.rain_condition = w.rain_condition
    AND t.hour = w.hour

ORDER BY
    CASE t.temperature_band
        WHEN 'Below 0 C' THEN 1
        WHEN '0-10 C' THEN 2
        WHEN '10-20 C' THEN 3
        WHEN '20-30 C' THEN 4
        WHEN '30+ C' THEN 5
        ELSE 6
    END,
    t.weather_condition,
    t.hour;